In [6]:
#isic_id,lesion_id,diagnosis_3,binary_label,label,image_path

from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/bachelor_thesis_data")

TRAIN_CSV = BASE_DIR / "splits/train_split.csv"

DIFF_DIR = BASE_DIR / "diffusion/sdxl_melanoma_lora_run1/generated_checkpoint_2000"

OUT_DOUBLE_CSV = BASE_DIR / "splits/train_diff_double.csv"
OUT_BALANCED_CSV = BASE_DIR / "splits/train_diff_balanced.csv"

train_df = pd.read_csv(TRAIN_CSV)

NUM_DOUBLE_SYNTHETIC = int((train_df["label"] == 1).sum())
NUM_BALANCED_SYNTHETIC = int((train_df["label"] == 0).sum()) - NUM_DOUBLE_SYNTHETIC



print(train_df.shape)
print(train_df["label"].value_counts())
print(train_df.columns.tolist())
print(f"Number for double synthetic: {NUM_DOUBLE_SYNTHETIC}")
print(f"Number for balanced synthetic: {NUM_BALANCED_SYNTHETIC}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(8204, 6)
label
0    7261
1     943
Name: count, dtype: int64
['isic_id', 'lesion_id', 'diagnosis_3', 'binary_label', 'label', 'image_path']
Number for double synthetic: 943
Number for balanced synthetic: 6318


In [8]:
def create_augmented_training_csv(
    real_train_df: pd.DataFrame,
    synthetic_dir: Path,
    checkpoint_name: str,
    output_csv: Path,
    num_synthetic: int = 943,
) -> pd.DataFrame:

    image_paths = [
    synthetic_dir / f"diffusion_melanoma_{i:05d}.png"
    for i in range(num_synthetic)
    ]

    if len(image_paths) < num_synthetic:
        raise ValueError(
            f"Only {len(image_paths)} images found in {synthetic_dir}, "
            f"but {num_synthetic} are required."
        )

    image_paths = image_paths[:num_synthetic]

    synthetic_rows = pd.DataFrame({
        "isic_id": [
            f"synthetic_{checkpoint_name}_{i:05d}"
            for i in range(num_synthetic)
        ],
        "lesion_id": [
            f"synthetic_{checkpoint_name}_{i:05d}"
            for i in range(num_synthetic)
        ],
        "diagnosis_3": ["Melanoma, NOS"] * num_synthetic,
        "binary_label": ["Melanoma"] * num_synthetic,
        "label": [1] * num_synthetic,
        "image_path": [str(path) for path in image_paths],
    })

    synthetic_rows = synthetic_rows[real_train_df.columns]

    augmented_df = pd.concat(
        [real_train_df, synthetic_rows],
        ignore_index=True
    )

    augmented_df.to_csv(output_csv, index=False)

    print(f"Saved: {output_csv}")
    print(f"Total rows: {len(augmented_df)}")
    print(augmented_df["label"].value_counts())

    return augmented_df

In [9]:
train_diff_double = create_augmented_training_csv(
    real_train_df=train_df,
    synthetic_dir=DIFF_DIR,
    checkpoint_name="diff",
    output_csv=OUT_DOUBLE_CSV,
    num_synthetic=NUM_DOUBLE_SYNTHETIC,
)

train_diff_balanced = create_augmented_training_csv(
    real_train_df=train_df,
    synthetic_dir=DIFF_DIR,
    checkpoint_name="diff",
    output_csv=OUT_BALANCED_CSV,
    num_synthetic=NUM_BALANCED_SYNTHETIC,
)

Saved: /content/drive/MyDrive/bachelor_thesis_data/splits/train_diff_double.csv
Total rows: 9147
label
0    7261
1    1886
Name: count, dtype: int64
Saved: /content/drive/MyDrive/bachelor_thesis_data/splits/train_diff_balanced.csv
Total rows: 14522
label
0    7261
1    7261
Name: count, dtype: int64
